# String Output Parser Demo

In [ ]:
# Output Parser StrOutputParser Demo
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# llm = OllamaLLM(model=os.getenv("OLLAMA_MODEL"), base_url=os.getenv("OLLAMA_BASE_URL")) # New Way
llm = init_chat_model(model=os.getenv("MODEL_NAME"), model_provider=os.getenv("MODEL_PROVIDER"), max_tokens=1000)

prompt = PromptTemplate(
    template="""
You are a helpful assistant.

Summarize the following product review in one short sentence.

Review:
{text}
""",
input_variables=["text"],
validate_template=True
)

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

review = """
I bought these wireless headphones last week and the sound quality is amazing.
Battery lasts almost two days and they are very comfortable.
However, the microphone quality during calls could be better.
"""

result = chain.invoke({"text": review})


print(result)

# JSON Output Parser Demo

In [ ]:
# Load environment variables
import os
from dotenv import load_dotenv
load_dotenv()

# LangChain imports
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser


# -----------------------------
# 1️⃣ Initialize the model
# -----------------------------
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)


# -----------------------------
# 2️⃣ Create JSON output parser
# -----------------------------
# This parser converts model output → Python dict
parser = JsonOutputParser()


# -----------------------------
# 3️⃣ Define the prompt template
# -----------------------------
# format_instructions tells the model how to structure the JSON
prompt = PromptTemplate(
    template="""
Analyze the following product review and extract key insights.

{format_instructions}

Return JSON with the following fields:
- sentiment: positive, negative, or neutral
- main_issue: main complaint or praise
- recommendation: true if the user recommends the product, otherwise false

Review:
{text}
""",
    input_variables=["text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)


# -----------------------------
# 4️⃣ Create LCEL chain
# -----------------------------
# Pipeline:
# Prompt → Model → JSON Parser
chain = prompt | model | parser


# -----------------------------
# 5️⃣ Example review input
# -----------------------------
review = """
The laptop performance is excellent and it runs very fast.
However, the battery drains quickly when playing games.
"""


# -----------------------------
# 6️⃣ Run the chain
# -----------------------------
result = chain.invoke({
    "text": review
})


# -----------------------------
# 7️⃣ Print structured result
# -----------------------------
print(result)

# Pydantic Output Parser Demo

In [ ]:
# Pydantic Output Parser Demo
import os
from dotenv import load_dotenv

load_dotenv()

# Import LangChain utilities
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

# Import Pydantic for schema definition
from pydantic import BaseModel, Field
from typing import Literal

# -----------------------------
# 1️⃣ Define the structured schema
# -----------------------------
# This schema represents the structured data we want to extract
class SupportTicket(BaseModel):
    issue_type: Literal["billing", "technical", "account"] = Field(
        description="Category of the customer's issue"
    )
    
    priority: Literal["low", "medium", "high"] = Field(
        description="Urgency level of the issue"
    )
    
    product: str = Field(
        description="Product or service mentioned by the customer"
    )
    
    summary: str = Field(
        description="Short summary of the customer's problem"
    )

# -----------------------------
# 2️⃣ Create the output parser
# -----------------------------
# This parser will convert LLM output → JSON → Pydantic object
parser = PydanticOutputParser(pydantic_object=SupportTicket)

# -----------------------------
# 3️⃣ Initialize the language model
# -----------------------------
# We use environment variables to configure the model
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)

# -----------------------------
# 4️⃣ Define the prompt template
# -----------------------------
# This prompt instructs the LLM to extract information in a structured format
prompt = PromptTemplate(
    template="""
You are an AI assistant that extracts structured support ticket data.

{format_instructions}

Customer message:
{text}
""",
    # The input variable "text" will be provided when we invoke the chain
    input_variables=["text"],
    # Partial variables are filled at prompt creation time. The LLM will receive the format instructions as part of the prompt.
    partial_variables={"format_instructions": parser.get_format_instructions()},
    validate_template=True
)

print("=============== PROMPT ===============")
print("parser format instructions:", parser.get_format_instructions())
print("==============================")

# -----------------------------
# 5️⃣ Build the LCEL chain
# -----------------------------
# Execution pipeline:
# Prompt → LLM → Pydantic Parser
chain = prompt | model | parser

# -----------------------------
# 6️⃣ Run the chain
# -----------------------------
# Example customer message
result = chain.invoke({
    "text": """
    Hi, I was charged twice for my Pro subscription this month.
    Please fix this immediately because the extra charge caused issues with my bank account.
    """
})

# -----------------------------
# 7️⃣ Print structured result
# -----------------------------
print("=============== RESULT ===============")
print(result)


# Access individual fields
print(result.issue_type)
print(result.priority)
print(result.product)
print(result.summary)
